# 03 ACT 在 ROCm 上的迁移与 DAgger 诊断

        这一节用 ACT 说明：训练 loss 下降不等于闭环 rollout 成功。大家会先看诊断曲线，再学习如何把数据回放、open-loop、closed-loop、失败视频和 DAgger 串成一条排障链路。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1：查看 ACT 诊断曲线


In [ ]:
snapshot = json.loads((ASSET_DIR / "metrics_snapshot.json").read_text(encoding="utf-8"))
rows = []
for item in snapshot["act_progress"]:
    rows.append((item["label"], f'{item["success"]}/{item["total"]}', f'{item["rate"]:.1%}'))
md_table(["阶段", "physical_success", "成功率"], rows)
show_asset("act_dagger_progress_curve.png", width=960)


这条曲线不是为了证明 ACT 已经完美泛化，而是为了展示排障方向：先确认数据和闭环状态，再决定是否需要 DAgger。


## Checkpoint 2：ACT 严格评估命令模板


In [ ]:
act_policy = MODEL_ROOT / "act_best" / "step_5000"
cmd = f"""
cd "$PROJECT_ROOT"
export PYTHONPATH="$PWD:${{PYTHONPATH:-}}"
./.venv/bin/python eval_policy_success.py \\
  --policy act \\
  --act-policy-path "{act_policy}" \\
  --physical-success \\
  --seed-start 1000 \\
  --episodes 10 \\
  --max-action-steps 1200 \\
  --output-jsonl "$OUTPUT_ROOT/act_seen_1000_1009.jsonl"
"""
print(cmd)


正式训练或批量评估更适合在终端或后台脚本里跑。Notebook 这里负责保存命令模板、解释参数含义和读取结果。


## Checkpoint 3：成功与失败视频对照


In [ ]:
show_asset("act_success_sequence.jpg", width=1100)
show_asset("act_failure_sequence.jpg", width=1100)


ACT 的失败常常不是完全不动，而是接近、夹取、搬运或释放中的某一段出问题。大家复核视频时要写出失败发生在哪个阶段。


## Checkpoint 4：记录 DAgger 实验


In [ ]:
rows = [
    ("prefix 长度", "例如 40 个 control tick"),
    ("oracle 类型", "scripted policy / human correction"),
    ("timestamp offset", "例如 correction episode 使用 2.0"),
    ("采样权重", "例如 correction episode weight=0.25"),
    ("评估 seed", "seen / mixed / heldout 分开写"),
    ("替换基线条件", "必须同一 physical_success 口径超过旧基线"),
]
md_table(["记录项", "示例"], rows)
